# Evaluate Fine-Tuned LoRA Adapter on Training Data

This notebook loads the fine-tuned Nemotron-3-Nano-30B LoRA adapter and evaluates it against the training set ground truth.

### Required Kaggle Inputs
1. **Competition data** — `nvidia-nemotron-model-reasoning-challenge` (for `train.csv`)
2. **Base model** — `metric/nemotron-3-nano-30b-a3b-bf16` (Transformers format)
3. **Your LoRA adapter** — `harrytsjan/nemotron-cot-0-7` (Kaggle dataset)
4. **Offline packages** — `mayukh18/nemotron-packages` (for Unsloth + mamba_ssm)

### What this notebook does
1. Installs dependencies (Unsloth, mamba_ssm)
2. Loads the base Nemotron model + your LoRA adapter
3. Runs inference on all (or a subset of) `train.csv`
4. Parses `\boxed{}` answers and compares against ground truth
5. Reports accuracy overall and by puzzle type

> **GPU required.** Set Accelerator to **GPU RTX Pro 6000** in notebook settings.

## Step 1: Install Dependencies

In [ ]:
!pip install -q --no-index --find-links /kaggle/input/datasets/mayukh18/nemotron-packages/packages unsloth trl peft transformers datasets accelerate bitsandbytes
!pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
!pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

## Step 2: Configuration

In [ ]:
import os
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

import re
import math
import time
from pathlib import Path
import pandas as pd
import torch
from unsloth import FastLanguageModel

# ======== PATHS ========
# Base model path (Kaggle model input)
MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"

# Your fine-tuned LoRA adapter (Kaggle dataset input)
ADAPTER_PATH = "/kaggle/input/datasets/harrytsjan/nemotron-cot-0-7"

# Training data with ground truth answers
TRAIN_CSV = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"

# ======== EVAL CONFIG ========
MAX_SEQ_LEN = 8192       # match competition setting
MAX_NEW_TOKENS = 7680    # match competition vLLM setting
TEMPERATURE = 0.0        # match competition: greedy decoding
EVAL_SUBSET = 10       # Set to an integer (e.g., 100) to evaluate a subset, or None for all
BATCH_DISPLAY = 50       # Print progress every N examples

# Prompt suffix (matches the competition metric)
BOXED_INSTRUCTION = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

print("Config ready.")
print(f"Adapter path: {ADAPTER_PATH}")
print(f"Adapter files: {os.listdir(ADAPTER_PATH)}")

## Step 3: Load Base Model + LoRA Adapter

In [ ]:
print("Loading base model + LoRA adapter...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH,           # Load directly from LoRA adapter dir
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
    trust_remote_code=True,
    unsloth_force_compile=False,
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    dtype=None,
)

# Switch to inference mode (Unsloth optimisation — faster generation)
FastLanguageModel.for_inference(model)

print(f"Model loaded on {model.device}")
print(f"Tokenizer vocab size: {len(tokenizer)}")

**If Unsloth cannot load the adapter directly**, uncomment the cell below to load base model first and then merge the adapter manually.

In [ ]:
# # === FALLBACK: Load base model first, then apply adapter ===
# from peft import PeftModel
#
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=MODEL_PATH,
#     max_seq_length=MAX_SEQ_LEN,
#     load_in_4bit=False,
#     trust_remote_code=True,
#     unsloth_force_compile=False,
#     attn_implementation="eager",
#     torch_dtype=torch.bfloat16,
# )
# model = PeftModel.from_pretrained(model, ADAPTER_PATH)
# FastLanguageModel.for_inference(model)
# print("Base model + adapter loaded via fallback.")

## Step 4: Load Training Data

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
print(f"Total training examples: {len(train_df):,}")
print(f"Columns: {list(train_df.columns)}")

if EVAL_SUBSET is not None:
    eval_df = train_df.sample(EVAL_SUBSET, random_state=42).reset_index(drop=True)
    print(f"\nEvaluating on random subset of {EVAL_SUBSET} examples.")
else:
    eval_df = train_df.reset_index(drop=True)
    print(f"\nEvaluating on ALL {len(eval_df):,} examples.")

print(f"\nSample prompt (first 200 chars): {eval_df['prompt'].iloc[0][:200]}...")
print(f"Sample answer: {eval_df['answer'].iloc[0]}")

## Step 5: Answer Extraction & Verification Helpers

In [ ]:
def extract_final_answer(text: str | None) -> str:
    r"""Extract the final answer from model output, prioritising \boxed{}."""
    if text is None:
        return "NOT_FOUND"

    # Prefer \boxed{...} — take the last one (model may refine its answer)
    matches = re.findall(r"\\boxed\{([^}]*)(?:\}|$)", text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        return non_empty[-1] if non_empty else matches[-1].strip()

    # Common fallback patterns
    patterns = [
        r"The final answer is:\s*([^\n]+)",
        r"Final answer is:\s*([^\n]+)",
        r"Final answer\s*[:ï¼]\s*([^\n]+)",
    ]
    for pattern in patterns:
        m = re.findall(pattern, text, re.IGNORECASE)
        if m:
            return m[-1].strip()

    # Last numeric value
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    if nums:
        return nums[-1]

    # Last non-empty line
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return lines[-1] if lines else "NOT_FOUND"


def verify(stored_answer: str, predicted: str) -> bool:
    """Return True if predicted matches stored_answer (numeric tolerance or exact string)."""
    stored_answer = stored_answer.strip()
    predicted = predicted.strip()
    # Exact string match first
    if predicted == stored_answer:
        return True
    # Case-insensitive string match
    if predicted.lower() == stored_answer.lower():
        return True
    # Numeric comparison with tolerance
    try:
        return math.isclose(float(stored_answer), float(predicted),
                            rel_tol=1e-2, abs_tol=1e-5)
    except (ValueError, OverflowError):
        return False


print("Helper functions ready.")

# Quick sanity check
assert extract_final_answer(r"The answer is \boxed{42}") == "42"
assert verify("42", "42.0")
assert verify("hello world", "Hello World")
print("Sanity checks passed.")

## Step 6: Run Inference & Evaluate

This runs single-example inference (no batching) to match competition behavior. 
For 9,500 examples with `max_new_tokens=7680`, this will take a long time. 
Consider setting `EVAL_SUBSET` to e.g. 100 or 500 for a quick estimate first.

In [ ]:
results = []
correct = 0
total = len(eval_df)
start_time = time.time()

print(f"Starting evaluation on {total} examples...")
print(f"Max new tokens: {MAX_NEW_TOKENS}, Temperature: {TEMPERATURE}")
print("=" * 70)

for idx, row in eval_df.iterrows():
    # Format prompt exactly like competition metric
    user_content = row["prompt"] + BOXED_INSTRUCTION
    messages = [{"role": "user", "content": user_content}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    input_ids = tokenizer(text, return_tensors="pt", truncation=True,
                          max_length=MAX_SEQ_LEN).input_ids.to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=None,       # greedy (temperature=0 equivalent)
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only newly generated tokens
    generated = tokenizer.decode(
        output_ids[0][input_ids.shape[1]:],
        skip_special_tokens=True,
    )

    predicted = extract_final_answer(generated)
    expected = str(row["answer"]).strip()
    is_correct = verify(expected, predicted)

    if is_correct:
        correct += 1

    results.append({
        "id": row["id"],
        "expected": expected,
        "predicted": predicted,
        "correct": is_correct,
        "generated_len": len(generated),
        "prompt_snippet": row["prompt"][:80],
    })

    # Progress reporting
    done = idx + 1
    if done % BATCH_DISPLAY == 0 or done == total:
        elapsed = time.time() - start_time
        rate = elapsed / done
        eta = rate * (total - done)
        acc_so_far = correct / done
        print(
            f"[{done:>5}/{total}]  "
            f"Accuracy: {acc_so_far:.2%} ({correct}/{done})  "
            f"Elapsed: {elapsed/60:.1f}m  "
            f"ETA: {eta/60:.1f}m  "
            f"({rate:.1f}s/example)"
        )

total_time = time.time() - start_time
print("=" * 70)
print(f"\nFINAL ACCURACY: {correct}/{total} = {correct/total:.4f} ({correct/total:.2%})")
print(f"Total time: {total_time/60:.1f} minutes ({total_time/3600:.1f} hours)")

## Step 7: Detailed Results Analysis

In [ ]:
results_df = pd.DataFrame(results)

print(f"Overall Accuracy: {results_df['correct'].mean():.4f} ({results_df['correct'].sum()}/{len(results_df)})")
print(f"\n{'='*70}")
print("\n--- Incorrect Predictions (first 20) ---\n")

wrong = results_df[~results_df["correct"]].head(20)
for _, r in wrong.iterrows():
    print(f"  ID: {r['id']}")
    print(f"  Prompt: {r['prompt_snippet']}...")
    print(f"  Expected:  {r['expected']}")
    print(f"  Predicted: {r['predicted']}")
    print(f"  Gen length: {r['generated_len']} chars")
    print()

In [ ]:
# Categorize puzzles by type based on prompt keywords
def classify_puzzle(prompt: str) -> str:
    prompt_lower = prompt.lower()
    if "bit manipulation" in prompt_lower or "8-bit binary" in prompt_lower:
        return "bit_manipulation"
    elif "encryption" in prompt_lower or "decrypt" in prompt_lower:
        return "encryption"
    elif "roman numeral" in prompt_lower:
        return "roman_numerals"
    elif "equation" in prompt_lower or "algebraic" in prompt_lower:
        return "algebraic"
    elif "gravity" in prompt_lower or "free fall" in prompt_lower:
        return "physics"
    elif "convert" in prompt_lower or "conversion" in prompt_lower:
        return "conversion"
    elif "sequence" in prompt_lower or "pattern" in prompt_lower:
        return "sequence"
    else:
        return "other"

results_df["puzzle_type"] = eval_df["prompt"].apply(classify_puzzle)

print("Accuracy by puzzle type:")
print("=" * 50)
type_stats = results_df.groupby("puzzle_type").agg(
    total=("correct", "count"),
    correct=("correct", "sum"),
    accuracy=("correct", "mean"),
).sort_values("accuracy", ascending=False)

for ptype, row in type_stats.iterrows():
    bar = "█" * int(row["accuracy"] * 30)
    print(f"  {ptype:<20s}  {row['correct']:>4.0f}/{row['total']:>4.0f}  {row['accuracy']:.2%}  {bar}")

print(f"\n  {'OVERALL':<20s}  {results_df['correct'].sum():>4}/{len(results_df):>4}  {results_df['correct'].mean():.2%}")

## Step 8: Save Results to CSV

In [ ]:
output_path = "/kaggle/working/evaluation_results.csv"
results_df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")
print(f"\nSummary:")
print(f"  Total evaluated: {len(results_df)}")
print(f"  Correct:         {results_df['correct'].sum()}")
print(f"  Accuracy:        {results_df['correct'].mean():.4f}")
print(f"  Predictions with NOT_FOUND: {(results_df['predicted'] == 'NOT_FOUND').sum()}")